In [ ]:
import tensorflow as tf
import numpy as np
import os
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape
import fiona
from fiona.crs import from_epsg
from multiprocessing import Pool
import tempfile
import shutil

# ======================= MODEL ARCHITECTURE =======================

def CBRR_conv_block(inputs, filters, dilation=1, name=None):
    conv1 = layers.Conv2D(filters, 3, padding='same', dilation_rate=dilation, 
                         name=f'{name}_conv1')(inputs)
    bn1 = layers.BatchNormalization(name=f'{name}_bn1')(conv1)
    relu1 = layers.ReLU(name=f'{name}_relu1')(bn1)
    
    conv2 = layers.Conv2D(filters, 3, padding='same', dilation_rate=dilation,
                         name=f'{name}_conv2')(relu1)
    bn2 = layers.BatchNormalization(name=f'{name}_bn2')(conv2)
    relu2 = layers.ReLU(name=f'{name}_relu2')(bn2)
    
    conv3 = layers.Conv2D(filters, 3, padding='same', dilation_rate=dilation,
                         name=f'{name}_conv3')(relu2)
    bn3 = layers.BatchNormalization(name=f'{name}_bn3')(conv3)
    relu3 = layers.ReLU(name=f'{name}_relu3')(bn3)
    
    return layers.Add(name=f'{name}_add')([relu3, relu1])

def MSCFF_V2(input_shape=(1024, 1024, 3)):
    inputs = tf.keras.Input(shape=input_shape, name='input')
    
    # Encoder
    c1 = CBRR_conv_block(inputs, 64, name='c1')
    p1 = layers.MaxPooling2D(2, name='p1')(c1)
    
    c2 = CBRR_conv_block(p1, 128, name='c2')
    p2 = layers.MaxPooling2D(2, name='p2')(c2)
    
    c3 = CBRR_conv_block(p2, 256, name='c3')
    p3 = layers.MaxPooling2D(2, name='p3')(c3)
    
    c4 = CBRR_conv_block(p3, 512, name='c4')
    c5 = CBRR_conv_block(c4, 512, dilation=2, name='c5')
    c6 = CBRR_conv_block(c5, 512, dilation=4, name='c6')
    
    # Decoder
    c7 = CBRR_conv_block(c6, 512, dilation=4, name='c7')
    a1 = layers.Add(name='a1')([c7, c6])
    
    c8 = CBRR_conv_block(a1, 512, dilation=2, name='c8')
    a2 = layers.Add(name='a2')([c8, c5])
    
    c9 = CBRR_conv_block(a2, 512, name='c9')
    a3 = layers.Add(name='a3')([c9, c4])
    
    # Upsampling path
    u1 = layers.Conv2DTranspose(256, 4, strides=2, padding='same', name='u1')(a3)
    c10 = CBRR_conv_block(u1, 256, name='c10')
    a4 = layers.Add(name='a4')([c10, c3])
    
    u2 = layers.Conv2DTranspose(128, 4, strides=2, padding='same', name='u2')(a4)
    c11 = CBRR_conv_block(u2, 128, name='c11')
    a5 = layers.Add(name='a5')([c11, c2])
    
    u3 = layers.Conv2DTranspose(64, 4, strides=2, padding='same', name='u3')(a5)
    c12 = CBRR_conv_block(u3, 64, name='c12')
    a6 = layers.Add(name='a6')([c12, c1])
    
    # Multi-scale Fusion
    ms1 = layers.Conv2DTranspose(64, 16, strides=8, padding='same', name='ms1')(c7)
    ms2 = layers.Conv2DTranspose(64, 16, strides=8, padding='same', name='ms2')(c8)
    ms3 = layers.Conv2DTranspose(64, 16, strides=8, padding='same', name='ms3')(c9)
    ms4 = layers.Conv2DTranspose(64, 8, strides=4, padding='same', name='ms4')(a4)
    ms5 = layers.Conv2DTranspose(64, 4, strides=2, padding='same', name='ms5')(a5)
    
    # Cloud/Shadow Separation
    cloud = layers.Conv2D(32, 1, activation='relu', name='cloud1')(ms4)
    cloud = layers.Conv2D(32, 3, padding='same', activation='relu', name='cloud2')(cloud)
    
    shadow = layers.Conv2D(32, 1, activation='relu', name='shadow1')(ms5)
    shadow = layers.Conv2D(32, 3, padding='same', activation='relu', name='shadow2')(shadow)
    
    # Concatenate and output
    concat = layers.Concatenate(axis=-1, name='concat')(
        [ms1, ms2, ms3, cloud, shadow, a6]
    )
    
    outputs = layers.Conv2D(3, 3, padding='same', name='output_conv')(concat)
    outputs = layers.Softmax(name='output_softmax')(outputs)
    
    return tf.keras.Model(inputs=inputs, outputs=outputs, name='MSCFF_V2')

# ======================= DATA PROCESSING & TRAINING =======================

class PatchGenerator:
    def __init__(self, patch_size=1024, overlap=128):
        self.patch_size = patch_size
        self.overlap = overlap
        self.stride = patch_size - overlap
        
    def split_image(self, image):
        patches = []
        positions = []
        h, w = image.shape[:2]
        
        for y in range(0, h, self.stride):
            for x in range(0, w, self.stride):
                y_end = min(y + self.patch_size, h)
                x_end = min(x + self.patch_size, w)
                
                patch = image[y:y_end, x:x_end]
                
                # Pad if necessary
                pad_y = self.patch_size - patch.shape[0]
                pad_x = self.patch_size - patch.shape[1]
                if pad_y > 0 or pad_x > 0:
                    patch = np.pad(
                        patch, 
                        ((0, pad_y), (0, pad_x), (0, 0)),
                        mode='reflect'
                    )
                
                patches.append(patch)
                positions.append((y, x))
        
        return np.array(patches), positions

    def stitch_patches(self, patches, positions, original_shape):
        h, w = original_shape[:2]
        output = np.zeros((h, w, 3), dtype=np.float32)
        count = np.zeros((h, w, 1), dtype=np.float32)
        
        for patch, (y, x) in zip(patches, positions):
            ph, pw = patch.shape[:2]
            actual_h = min(ph, h - y)
            actual_w = min(pw, w - x)
            
            output[y:y+actual_h, x:x+actual_w] += patch[:actual_h, :actual_w]
            count[y:y+actual_h, x:x+actual_w] += 1
        
        # Avoid division by zero
        count[count == 0] = 1
        return output / count

def prepare_training_data(image_dir, mask_dir, output_dir, patch_size=1024, overlap=128):
    os.makedirs(output_dir, exist_ok=True)
    patch_gen = PatchGenerator(patch_size, overlap)
    
    for name in os.listdir(image_dir):
        if not name.endswith('.tif'):
            continue
            
        # Load image and mask
        with rasterio.open(os.path.join(image_dir, name)) as src:
            img = src.read().transpose(1, 2, 0).astype(np.float32) / 65535.0
            meta = src.meta.copy()
        
        with rasterio.open(os.path.join(mask_dir, name)) as src:
            mask = src.read(1).astype(np.uint8)
        
        # Create patches
        img_patches, img_pos = patch_gen.split_image(img)
        mask_patches, mask_pos = patch_gen.split_image(mask[..., np.newaxis])
        
        # Save as TFRecord
        record_file = os.path.join(output_dir, f"{os.path.splitext(name)[0]}.tfrecord")
        with tf.io.TFRecordWriter(record_file) as writer:
            for i, (img_patch, mask_patch) in enumerate(zip(img_patches, mask_patches)):
                example = tf.train.Example(features=tf.train.Features(feature={
                    'image': tf.train.Feature(
                        bytes_list=tf.train.BytesList(value=[img_patch.tobytes()])),
                    'mask': tf.train.Feature(
                        bytes_list=tf.train.BytesList(value=[mask_patch.tobytes()])),
                    'height': tf.train.Feature(
                        int64_list=tf.train.Int64List(value=[img_patch.shape[0]])),
                    'width': tf.train.Feature(
                        int64_list=tf.train.Int64List(value=[img_patch.shape[1]]))
                }))
                writer.write(example.SerializeToString())

def parse_tfrecord(example, patch_size=1024):
    features = {
        'image': tf.io.FixedLenFeature([], tf.string),
        'mask': tf.io.FixedLenFeature([], tf.string),
        'height': tf.io.FixedLenFeature([], tf.int64),
        'width': tf.io.FixedLenFeature([], tf.int64)
    }
    example = tf.io.parse_single_example(example, features)
    
    height = tf.cast(example['height'], tf.int32)
    width = tf.cast(example['width'], tf.int32)
    
    image = tf.io.decode_raw(example['image'], tf.float32)
    image = tf.reshape(image, (height, width, 3))
    
    mask = tf.io.decode_raw(example['mask'], tf.uint8)
    mask = tf.reshape(mask, (height, width, 1))
    mask = tf.one_hot(tf.squeeze(mask, -1), depth=3)
    
    return image, mask

def create_dataset(tfrecord_dir, batch_size=8, patch_size=1024):
    tfrecords = [os.path.join(tfrecord_dir, f) for f in os.listdir(tfrecord_dir) 
                if f.endswith('.tfrecord')]
    dataset = tf.data.TFRecordDataset(tfrecords)
    dataset = dataset.map(
        lambda x: parse_tfrecord(x, patch_size), 
        num_parallel_calls=tf.data.AUTOTUNE
    )
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def train_model(train_dir, val_dir, model_save_path, epochs=30, batch_size=8):
    train_ds = create_dataset(train_dir, batch_size)
    val_ds = create_dataset(val_dir, batch_size)
    
    model = MSCFF_V2()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.OneHotMeanIoU(num_classes=3, name='mean_iou'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )
    
    callbacks = [
        tf.keras.callbacks.EarlyStopping(patience=5, monitor='val_mean_iou', mode='max'),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=model_save_path,
            save_best_only=True,
            monitor='val_loss'
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6)
    ]
    
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks
    )

# ======================= INFERENCE & POST-PROCESSING =======================

def predict_large_image(model, image_path, output_dir, patch_size=1024, overlap=128):
    os.makedirs(output_dir, exist_ok=True)
    patch_gen = PatchGenerator(patch_size, overlap)
    
    with rasterio.open(image_path) as src:
        img = src.read().transpose(1, 2, 0)
        meta = src.meta.copy()
        meta.update(dtype=rasterio.uint8, count=1, nodata=0)
        
        # Convert to float and normalize
        img = img.astype(np.float32) / 65535.0
        
        # Generate patches
        patches, positions = patch_gen.split_image(img)
    
    # Predict in batches
    predictions = []
    batch_size = 4
    
    for i in range(0, len(patches), batch_size):
        batch = patches[i:i+batch_size]
        preds = model.predict(batch, verbose=0)
        predictions.extend(preds)
    
    # Stitch predictions
    class_map = np.argmax(np.array(predictions), axis=-1)
    stitched = patch_gen.stitch_patches(
        class_map, 
        positions, 
        (img.shape[0], img.shape[1], 1)
    )
    stitched = np.squeeze(stitched).astype(np.uint8)
    
    # Save class map
    class_map_path = os.path.join(output_dir, 'class_map.tif')
    with rasterio.open(class_map_path, 'w', **meta) as dst:
        dst.write(stitched, 1)
    
    # Generate masks and shapefiles
    generate_shapefiles(stitched, meta, output_dir)
    
    return class_map_path

def generate_shapefiles(class_map, meta, output_dir):
    # Create masks
    cloud_mask = (class_map == 1).astype(np.uint8)
    shadow_mask = (class_map == 2).astype(np.uint8)
    
    # Save mask files
    with rasterio.open(os.path.join(output_dir, 'cloud_mask.tif'), 'w', **meta) as dst:
        dst.write(cloud_mask, 1)
    
    with rasterio.open(os.path.join(output_dir, 'shadow_mask.tif'), 'w', **meta) as dst:
        dst.write(shadow_mask, 1)
    
    # Generate polygons in parallel
    with Pool() as pool:
        cloud_polygons = pool.starmap(extract_polygons, [(cloud_mask, meta['transform'])])
        shadow_polygons = pool.starmap(extract_polygons, [(shadow_mask, meta['transform'])])
    
    # Save shapefiles
    save_shapefile(cloud_polygons[0], os.path.join(output_dir, 'cloud_polygons.shp'), 'cloud')
    save_shapefile(shadow_polygons[0], os.path.join(output_dir, 'shadow_polygons.shp'), 'shadow')

def extract_polygons(mask, transform):
    return [
        (shape(geom), value)
        for geom, value in shapes(mask, transform=transform)
        if value == 1
    ]

def save_shapefile(polygons, path, feature_type):
    schema = {
        'geometry': 'Polygon',
        'properties': {'class': 'int'}
    }
    
    with fiona.open(path, 'w', 
                   driver='ESRI Shapefile',
                   schema=schema,
                   crs=from_epsg(4326)) as dst:
        for geom, value in polygons:
            dst.write({
                'geometry': geom.__geo_interface__,
                'properties': {'class': value}
            })

# ======================= MAIN EXECUTION =======================

if __name__ == "__main__":
    # Configuration
    TRAIN_IMAGE_DIR = "/path/to/train/images"
    TRAIN_MASK_DIR = "/path/to/train/masks"
    VAL_IMAGE_DIR = "/path/to/val/images"
    VAL_MASK_DIR = "/path/to/val/masks"
    PATCH_DIR = "/path/to/patches"
    MODEL_PATH = "/path/to/model.keras"
    
    # Step 1: Prepare training data
    prepare_training_data(TRAIN_IMAGE_DIR, TRAIN_MASK_DIR, 
                         os.path.join(PATCH_DIR, "train"))
    prepare_training_data(VAL_IMAGE_DIR, VAL_MASK_DIR,
                         os.path.join(PATCH_DIR, "val"))
    
    # Step 2: Train model
    train_model(
        os.path.join(PATCH_DIR, "train"),
        os.path.join(PATCH_DIR, "val"),
        MODEL_PATH,
        epochs=30,
        batch_size=8
    )
    
    # Step 3: Inference on new image
    model = tf.keras.models.load_model(MODEL_PATH)
    predict_large_image(
        model,
        "/path/to/test_image.tif",
        "/path/to/output",
        patch_size=1024,
        overlap=128
    )